# 03 - Paper deep-learning classifiers

Clean versions of the large-dataset LSTM, Bidirectional LSTM, CNN+LSTM, deeper CNN+LSTM, and Conv2D experiments.

Paper Table 1.3 reports:

- CNN + LSTM: 98.1%
- deeper CNN + LSTM: 97.8%
- Bidirectional LSTM: 96.4%
- Conv2D: 94.1%
- LSTM: 82.5%

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

sys.path.append("..")
from src.labels import fine_label_ids_in_original_order

DATA_PATH = Path("../data/new_features_3sec.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "The recovered 80,207-row new_features_3sec.csv is missing. "
        "See data/README.md and notebook 01 to regenerate it from the original audio."
    )

data = pd.read_csv(DATA_PATH)
y, mapping = fine_label_ids_in_original_order(data["filename"])
X_df = data.drop(columns=["filename", "length"])

print("Table shape:", data.shape)
print("Numeric features:", X_df.shape[1])
print("Fine-grained labels:", len(mapping))

In [ ]:
# Historical preprocessing preserved from the recovered notebook:
# MinMaxScaler was fit before the split and no random_state was stored.
scaler = MinMaxScaler()
X = scaler.fit_transform(X_df)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2
)

print("Train:", X_train.shape, "Test:", X_test.shape)

In [ ]:
import tensorflow as tf
from tensorflow.keras.utils import to_categorical

from src.deep_models import (
    build_lstm,
    build_bidirectional_lstm,
    build_cnn_lstm,
    build_deep_cnn_bilstm,
    build_conv2d,
)

n_features = X_train.shape[1]
n_classes = len(mapping)

print("n_features =", n_features)
print("n_classes =", n_classes)

## Simple LSTM

The model evaluates to approximately **82.53%**, matching the paper's rounded **82.5%** row.

In [ ]:
X_train_step = X_train.reshape((-1, 1, n_features))
X_test_step = X_test.reshape((-1, 1, n_features))

lstm_model = build_lstm(n_features, n_classes)

# Expensive historical-style run:
# lstm_model.fit(
#     X_train_step, y_train,
#     epochs=500, batch_size=32,
#     validation_data=(X_test_step, y_test)
# )

## Bidirectional LSTM

Held-out output: **96.440595%**, matching the paper's **96.4%** row.

In [ ]:
X_train_seq = X_train.reshape((-1, n_features, 1))
X_test_seq = X_test.reshape((-1, n_features, 1))

y_train_oh = to_categorical(y_train, num_classes=n_classes)
y_test_oh = to_categorical(y_test, num_classes=n_classes)

bilstm_model = build_bidirectional_lstm(n_features, n_classes)

# Historical-style run:
# bilstm_model.fit(
#     X_train_seq, y_train_oh,
#     epochs=100, batch_size=32,
#     validation_data=(X_test_seq, y_test_oh)
# )

## CNN + LSTM

In [ ]:
cnn_lstm = build_cnn_lstm(n_features, n_classes)

# cnn_lstm.fit(
#     X_train_seq, y_train_oh,
#     epochs=200, batch_size=32,
#     validation_data=(X_test_seq, y_test_oh)
# )

## Deeper CNN + Bidirectional LSTM

The training run reaches approximately **97.83% validation accuracy**, consistent with the paper's 97.8% row.

In [ ]:
deep_model = build_deep_cnn_bilstm(n_features, n_classes)

# deep_model.fit(
#     X_train_seq, y_train_oh,
#     epochs=50, batch_size=32,
#     validation_data=(X_test_seq, y_test_oh)
# )

## Conv2D model

In [ ]:
X_train_2d = X_train.reshape((-1, n_features, 1, 1))
X_test_2d = X_test.reshape((-1, n_features, 1, 1))

conv2d_model = build_conv2d(n_features, n_classes)

# conv2d_model.fit(
#     X_train_2d, y_train_oh,
#     epochs=200, batch_size=32,
#     validation_data=(X_test_2d, y_test_oh)
# )